# FedDiff-MSA: Federated Diffusion-based Multimodal Sentiment Analysis

This notebook runs the full experiment suite on Google Colab with GPU.

**Setup:** Runtime → Change runtime type → **T4 GPU**

| Experiment | Est. Time (T4) | Paper Table |
|-----------|---------------|-------------|
| Main results | ~8h | Table 3 |
| Recovery quality | ~50min | Table 4 |
| Ablation study | ~10h | Table 5 |
| Privacy-utility | ~5h | Figure 6 |
| MIA | ~2.5h | Table 6 |
| Scalability | ~3h | Table 7 |
| Figures | ~5h | Fig 2-7 |

> **Note:** 12-hour session limit. Run experiments separately if needed. Results are saved to CSV/JSON and persist across sessions via Google Drive mount.

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Mount Google Drive (to save results persistently across sessions)
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")

In [ ]:
# Install git-lfs before cloning
!apt-get update -qq && apt-get install -y -qq git-lfs
!git lfs install
print("git-lfs installed.")

In [ ]:
# Clone the repository (with retry)
%cd /content
!rm -rf FedDiff-MSA

import subprocess
clone_ok = False
for attempt in range(3):
    print(f"Clone attempt {attempt+1}/3...")
    result = subprocess.run(
        ["git", "clone", "https://github.com/yjyjss/FedDiff-MSA.git"],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode == 0:
        print("Clone successful!")
        clone_ok = True
        break
    else:
        print(f"Failed: {result.stderr[:300]}")
        
if not clone_ok:
    print("\ngit clone failed. Trying GitHub API download as fallback...")
    import urllib.request, zipfile, io, os
    url = "https://github.com/yjyjss/FedDiff-MSA/archive/refs/heads/main.zip"
    print(f"Downloading zip from {url}...")
    urllib.request.urlretrieve(url, "/content/repo.zip")
    with zipfile.ZipFile("/content/repo.zip", 'r') as z:
        z.extractall("/content/")
    os.rename("/content/FedDiff-MSA-main", "/content/FedDiff-MSA")
    print("Downloaded via zip fallback.")

%cd /content/FedDiff-MSA
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Pull LFS files (the .npy data files, ~170MB total)
# If git lfs pull fails, download .npy files directly via GitHub API
import os, subprocess

data_dir = "data/features"
os.makedirs(data_dir, exist_ok=True)

# Check if LFS files are already downloaded (not LFS pointers)
def is_real_npy(path):
    if not os.path.exists(path):
        return False
    return os.path.getsize(path) > 10000  # LFS pointers are ~130 bytes

npy_files = ['mosei_text.npy', 'mosei_audio.npy', 'mosei_visual.npy', 'mosei_labels.npy']
all_present = all(is_real_npy(os.path.join(data_dir, f)) for f in npy_files)

if all_present:
    print("Data files already present, skipping download.")
else:
    # Try git lfs pull first
    result = subprocess.run(["git", "lfs", "pull"], capture_output=True, text=True, timeout=300)
    if result.returncode == 0 and all(is_real_npy(os.path.join(data_dir, f)) for f in npy_files):
        print("LFS pull successful!")
    else:
        print("git lfs pull failed or incomplete. Downloading .npy files directly...")
        # Download each .npy file directly from GitHub LFS API
        import urllib.request
        for fname in npy_files:
            fpath = os.path.join(data_dir, fname)
            if is_real_npy(fpath):
                print(f"  {fname}: already present ({os.path.getsize(fpath)/1e6:.1f} MB)")
                continue
            url = f"https://github.com/yjyjss/FedDiff-MSA/raw/main/data/features/{fname}"
            print(f"  Downloading {fname} from {url}...")
            urllib.request.urlretrieve(url, fpath)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"    Done: {size_mb:.1f} MB")

# Verify all data files
print("\n=== Data files verification ===")
for f in npy_files:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        status = "OK" if size_mb > 0.1 else "LFS POINTER (not real data!)"
        print(f"  {f}: {size_mb:.1f} MB  [{status}]")
    else:
        print(f"  {f}: MISSING!")

In [ ]:
# Install Python dependencies
!pip install -q scikit-learn matplotlib
print("Dependencies installed.")

In [ ]:
# Quick test to verify code works (tiny synthetic data, ~1 min)
!python run_experiments.py --test --exp main 2>&1 | tail -15

## 2. Configure Output Directory

Results will be saved to Google Drive so they persist across Colab sessions.

In [ ]:
import os

# Save results to Google Drive (persistent across sessions)
OUTPUT_DIR = '/content/drive/MyDrive/FedDiff-MSA-Results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Also create a symlink so the code writes there
!rm -rf outputs
!ln -s {OUTPUT_DIR} outputs

print(f"Results will be saved to: {OUTPUT_DIR}")
print(f"Symlink: outputs -> {OUTPUT_DIR}")

## 3. Run Experiments

Run each experiment separately. Each cell is independent — if your session disconnects, just re-run the setup cells above and continue with the next experiment.

### 3.1 Main Experiment → Table 3
9 methods × Setting C. ~8 hours on T4.

Run in 3 batches to avoid session timeout. Results are merged automatically.

In [ ]:
# Batch 1/3: Centralized, FedAvg, FedAvg+ZeroPad (~2.5h)
!python run_experiments.py --dataset mosei --setting C --device cuda --exp main --methods 'Centralized,FedAvg,FedAvg+ZeroPad' 2>&1 | tee outputs/log_main_b1.txt

In [ ]:
# Batch 2/3: FedAvg+MeanFill, FedProx, FedAvg+AutoEncoder (~2.5h)
!python run_experiments.py --dataset mosei --setting C --device cuda --exp main --methods 'MeanFill,FedProx,AutoEncoder' 2>&1 | tee outputs/log_main_b2.txt

In [ ]:
# Batch 3/3: FedMM-SA, Qiu et al., FedDiff-MSA (~2.5h)
!python run_experiments.py --dataset mosei --setting C --device cuda --exp main --methods 'FedMM-SA,Qiu,FedDiff-MSA' 2>&1 | tee outputs/log_main_b3.txt

### 3.2 Recovery Quality → Table 4
~50 minutes on T4.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp recovery 2>&1 | tee outputs/log_recovery.txt

### 3.3 Ablation Study → Table 5
11 variants. ~10 hours on T4.

Run in 3 batches. Results are merged automatically.

In [ ]:
# Batch 1/3: full, w/o Diffusion, w/o L_emo, w/o L_contrast (~3.5h)
!python run_experiments.py --dataset mosei --setting C --device cuda --exp ablation --variants 'full,w/o Diffusion,w/o L_emo,w/o L_contrast' 2>&1 | tee outputs/log_ablation_b1.txt

In [ ]:
# Batch 2/3: w/o MAFA-Diff, w/o Gradient, w/o Confidence, w/o Layered DP (~3.5h)
!python run_experiments.py --dataset mosei --setting C --device cuda --exp ablation --variants 'w/o MAFA-Diff,w/o Gradient,w/o Confidence,w/o Layered' 2>&1 | tee outputs/log_ablation_b2.txt

In [ ]:
# Batch 3/3: w/o Sparsification, w/ Decoupled, w/ Uniform DP budget (~3h)
!python run_experiments.py --dataset mosei --setting C --device cuda --exp ablation --variants 'w/o Sparsification,w/ Decoupled,w/ Uniform DP' 2>&1 | tee outputs/log_ablation_b3.txt

### 3.4 Privacy-Utility Trade-off → Figure 6
Layered DP vs Uniform DP across epsilon. ~5 hours on T4.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp privacy 2>&1 | tee outputs/log_privacy.txt

### 3.5 MIA Success Rates → Table 6
~2.5 hours on T4.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp mia 2>&1 | tee outputs/log_mia.txt

### 3.6 Client Scalability → Table 7
~3 hours on T4.

In [ ]:
!python run_experiments.py --dataset mosei --setting C --device cuda --exp scalability 2>&1 | tee outputs/log_scalability.txt

## 4. Generate Figures

Run after experiments are complete. Each figure reads from CSV/JSON results.

In [ ]:
# Generate all figures (reads from existing experiment results)
!python generate_figures.py --device cuda 2>&1 | tee outputs/log_figures.txt

Or generate individual figures:

In [ ]:
# Individual figures
# !python generate_figures.py --device cuda --fig convergence
# !python generate_figures.py --device cuda --fig tsne
# !python generate_figures.py --device cuda --fig noniid
# !python generate_figures.py --device cuda --fig missing_ratio
# !python generate_figures.py --device cuda --fig privacy_utility
# !python generate_figures.py --device cuda --fig contribution

## 5. View Results

In [ ]:
# List all output files
import os
output_dir = '/content/drive/MyDrive/FedDiff-MSA-Results'

print("=== CSV Table Files ===")
for f in sorted(os.listdir(output_dir)):
    if f.endswith('.csv'):
        size = os.path.getsize(os.path.join(output_dir, f)) / 1024
        print(f"  {f:45s} {size:.1f} KB")

print("\n=== JSON Detail Files ===")
for f in sorted(os.listdir(output_dir)):
    if f.endswith('.json'):
        size = os.path.getsize(os.path.join(output_dir, f)) / 1024
        print(f"  {f:45s} {size:.1f} KB")

print("\n=== Figures ===")
fig_dir = os.path.join(output_dir, 'figures')
if os.path.exists(fig_dir):
    for f in sorted(os.listdir(fig_dir)):
        if f.endswith('.pdf') or f.endswith('.png'):
            size = os.path.getsize(os.path.join(fig_dir, f)) / 1024
            print(f"  {f:45s} {size:.1f} KB")

In [ ]:
# Display a sample CSV result
import pandas as pd

csv_files = [f for f in os.listdir(output_dir) if f.endswith('.csv') and f.startswith('table_')]
if csv_files:
    print(f"Available tables: {csv_files}")
    print()
    # Show the first one
    df = pd.read_csv(os.path.join(output_dir, csv_files[0]), comment='#')
    print(f"=== {csv_files[0]} ===")
    print(df.to_string())

In [ ]:
# Display figures
from IPython.display import Image, display
import os

fig_dir = os.path.join(output_dir, 'figures')
if os.path.exists(fig_dir):
    png_files = sorted([f for f in os.listdir(fig_dir) if f.endswith('.png')])
    for f in png_files:
        print(f"\n=== {f} ===")
        display(Image(os.path.join(fig_dir, f)))
else:
    print("No figures found. Run the figure generation cell first.")

## 6. Download Results

Results are already in your Google Drive (`FedDiff-MSA-Results/`). You can also download specific files:

In [ ]:
# Download all CSV results as a zip
!cd /content/drive/MyDrive/FedDiff-MSA-Results && zip -j /content/results.zip *.csv *.json *.md 2>/dev/null

from google.colab import files
files.download('/content/results.zip')